In [40]:
import pandas as pd

df = pd.read_parquet("/home/nabin2004/Desktop/projects/lichess/data/processed/2013-01.parquet")
print(df.shape, df.columns.tolist())

(121332, 18) ['event', 'site', 'date', 'round', 'white', 'black', 'result', 'utc_date', 'utc_time', 'white_elo', 'black_elo', 'white_rating_diff', 'black_rating_diff', 'eco', 'opening', 'time_control', 'termination', 'moves']


In [41]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 121332 entries, 0 to 121331
Data columns (total 18 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   event              121332 non-null  str    
 1   site               121332 non-null  str    
 2   date               121332 non-null  str    
 3   round              121332 non-null  str    
 4   white              121332 non-null  str    
 5   black              121332 non-null  str    
 6   result             121332 non-null  str    
 7   utc_date           121332 non-null  str    
 8   utc_time           121332 non-null  str    
 9   white_elo          121254 non-null  float64
 10  black_elo          121192 non-null  float64
 11  white_rating_diff  121114 non-null  float64
 12  black_rating_diff  121114 non-null  float64
 13  eco                121332 non-null  str    
 14  opening            121332 non-null  str    
 15  time_control       121332 non-null  str    
 16  termination  

In [46]:
df[["event_name", "tournament_url"]] = (
    df["event"]
    .astype("string")
    .str.extract(
        r"^(?P<event_name>.*?)(?:\s+(?P<tournament_url>https://\S+))?$"
    )
)

df["event_name"] = df["event_name"].str.strip()

In [47]:
df['event_name'].value_counts()

event_name
Rated Blitz game              45388
Rated Classical game          41772
Rated Bullet game             32691
Rated Blitz tournament          896
Rated Bullet tournament         295
Rated Correspondence game       266
Rated Classical tournament       24
Name: count, dtype: int64[pyarrow]

In [49]:
time_control_map = {
    "Rated Blitz game": "Blitz",
    "Rated Blitz tournament": "Blitz",
    "Rated Bullet game": "Bullet",
    "Rated Bullet tournament": "Bullet",
    "Rated Classical game": "Classical",
    "Rated Classical tournament": "Classical",
    "Rated Correspondence game": "Correspondence",
}

time_control_order = ["Bullet", "Blitz", "Classical", "Correspondence"]

df["time_control"] = (
    df["event_name"]
    .map(time_control_map)
    .astype(pd.CategoricalDtype(categories=time_control_order, ordered=True))
)

df["is_tournament"] = df["event_name"].str.contains("tournament", case=False)


In [52]:
df['time_control'].value_counts()

time_control
Blitz             46284
Classical         41796
Bullet            32986
Correspondence      266
Name: count, dtype: int64

In [55]:
df['round'].value_counts()

round
?    121332
Name: count, dtype: int64

In [57]:
df['black']

0                   mamalak
1                 savinka59
2         VanillaShamanilla
3                       800
4          Naitero_Nagasaki
                ...        
121327             netsah08
121328          kualalumpur
121329          Richard_XII
121330             shueardm
121331           Tortfeasor
Name: black, Length: 121332, dtype: str

In [59]:
df['result'].unique()

<ArrowStringArray>
['1-0', '0-1', '1/2-1/2']
Length: 3, dtype: str

In [60]:
import pandas as pd
import numpy as np

# 1. Multi-class target (for classification)
# Map results to numeric labels
result_map = {
    "1-0": 0,      # White wins
    "0-1": 1,      # Black wins  
    "1/2-1/2": 2   # Draw
}
df["result_label"] = df["result"].map(result_map)

# 2. Binary targets (useful for separate models or imbalance handling)
df["white_win"] = (df["result"] == "1-0").astype(int)
df["black_win"] = (df["result"] == "0-1").astype(int)
df["is_draw"] = (df["result"] == "1/2-1/2").astype(int)

# 3. Winner indicator (from White's perspective)
# -1: Black wins, 0: Draw, 1: White wins
winner_from_white_perspective = {
    "1-0": 1,
    "0-1": -1,
    "1/2-1/2": 0
}
df["winner_white_perspective"] = df["result"].map(winner_from_white_perspective)

# 4. Decisive game flag (not a draw)
df["decisive_game"] = (~df["is_draw"]).astype(int)

# 5. For PyArrow string arrays, you can also extract directly
# (if you need to avoid pandas string methods)
df["result_str"] = df["result"].astype(str)
df["white_result"] = df["result_str"].str[0].astype(int)  # '1' for white
df["black_result"] = df["result_str"].str[-1].astype(int) # '1' for black

In [62]:
df['utc_date']

0         2012.12.31
1         2012.12.31
2         2012.12.31
3         2012.12.31
4         2012.12.31
             ...    
121327    2013.01.31
121328    2013.01.31
121329    2013.01.31
121330    2013.01.31
121331    2013.01.31
Name: utc_date, Length: 121332, dtype: str

In [63]:
import pandas as pd
import numpy as np

# 1. Convert string dates to datetime
df["utc_date"] = pd.to_datetime(df["utc_date"], format="%Y.%m.%d")

# 2. Extract temporal features for modeling
df["year"] = df["utc_date"].dt.year
df["month"] = df["utc_date"].dt.month
df["day"] = df["utc_date"].dt.day
df["day_of_week"] = df["utc_date"].dt.dayofweek  # Monday=0, Sunday=6
df["quarter"] = df["utc_date"].dt.quarter
df["day_of_year"] = df["utc_date"].dt.dayofyear

# 3. Weekend indicator (players might behave differently on weekends)
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

# 4. Month as ordered categorical (seasonal effects)
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
df["month_name"] = pd.Categorical(
    df["utc_date"].dt.strftime('%b'),
    categories=month_names,
    ordered=True
)

# 5. Days since start (numeric trend feature)
df["days_since_start"] = (df["utc_date"] - df["utc_date"].min()).dt.days

# ═══════════════════════════════════════════════════════
# TEMPORAL TRAIN-TEST SPLIT
# ═══════════════════════════════════════════════════════

# Sort by date first (critical for temporal split)
df = df.sort_values("utc_date").reset_index(drop=True)

In [64]:
df

,event,site,date,round,white,black,result,utc_date,utc_time,white_elo,...,black_result,year,month,day,day_of_week,quarter,day_of_year,is_weekend,month_name,days_since_start
0,Rated Classical game,https://lichess.org/j1dkb5dw,????.??.??,?,BFG9k,mamalak,1-0,2012-12-31,23:01:03,1639.0,...,0,2012,12,31,0,4,366,0,Dec,0
1,Rated Bullet game,https://lichess.org/4snemu9c,????.??.??,?,schutzstaffel,giao,1-0,2012-12-31,23:46:57,1827.0,...,0,2012,12,31,0,4,366,0,Dec,0
2,Rated Classical game,https://lichess.org/wb4ea59s,????.??.??,?,metrolog,vadi,1-0,2012-12-31,23:46:41,1638.0,...,0,2012,12,31,0,4,366,0,Dec,0
3,Rated Bullet game,https://lichess.org/x0ekc8d4,????.??.??,?,Boss92,Gardo,1-0,2012-12-31,23:46:38,1503.0,...,0,2012,12,31,0,4,366,0,Dec,0
4,Rated Bullet game,https://lichess.org/xfnan2k5,????.??.??,?,1000,giao,1-0,2012-12-31,23:46:36,1656.0,...,0,2012,12,31,0,4,366,0,Dec,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121327,Rated Bullet game,https://lichess.org/g53wqw4l,????.??.??,?,Krieg,Greenlan,1/2-1/2,2013-01-31,12:19:13,1652.0,...,2,2013,1,31,3,1,31,0,Jan,31
121328,Rated Classical game,https://lichess.org/40lenlrp,????.??.??,?,shorterg,Betta19731973,0-1,2013-01-31,12:20:05,1632.0,...,1,2013,1,31,3,1,31,0,Jan,31
121329,Rated Bullet game,https://lichess.org/7kscduxa,????.??.??,?,jansom,NoPlayNoFun,1-0,2013-01-31,12:20:33,1522.0,...,0,2013,1,31,3,1,31,0,Jan,31
121330,Rated Bullet game,https://lichess.org/dk8iotdb,????.??.??,?,jansom,pikaron,1-0,2013-01-31,12:15:58,1504.0,...,0,2013,1,31,3,1,31,0,Jan,31


In [65]:
df['utc_time']

0         23:01:03
1         23:46:57
2         23:46:41
3         23:46:38
4         23:46:36
            ...   
121327    12:19:13
121328    12:20:05
121329    12:20:33
121330    12:15:58
121331    22:59:31
Name: utc_time, Length: 121332, dtype: str

In [67]:
import pandas as pd
import numpy as np

# 1. Convert time strings to datetime
time_dt = pd.to_datetime(df["utc_time"], format="%H:%M:%S")

# 2. Extract basic components
df["hour"] = time_dt.dt.hour
df["minute"] = time_dt.dt.minute
df["second"] = time_dt.dt.second

# 3. Continuous time feature (seconds since midnight)
df["seconds_since_midnight"] = df["hour"] * 3600 + df["minute"] * 60 + df["second"]

# 4. Time of day category
df["time_of_day"] = pd.cut(
    df["hour"],
    bins=[-1, 6, 12, 18, 24],
    labels=["Night", "Morning", "Afternoon", "Evening"]
)

# 5. Binary time indicators
df["is_night"] = ((df["hour"] >= 22) | (df["hour"] <= 5)).astype(int)
df["is_morning"] = ((df["hour"] >= 6) & (df["hour"] <= 11)).astype(int)
df["is_afternoon"] = ((df["hour"] >= 12) & (df["hour"] <= 17)).astype(int)
df["is_evening"] = ((df["hour"] >= 18) & (df["hour"] <= 21)).astype(int)
df["is_peak_gaming"] = ((df["hour"] >= 16) & (df["hour"] <= 23)).astype(int)

# 6. Cyclical encoding for hour (handles 23→0 wrap-around)
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

# 7. Create combined datetime - FIX: convert datetime to string first
df["utc_datetime"] = pd.to_datetime(
    df["utc_date"].dt.strftime("%Y.%m.%d") + " " + df["utc_time"],
    format="%Y.%m.%d %H:%M:%S"
)

# Quick summary
print("Time distribution by period:")
print(df["time_of_day"].value_counts())
print(f"\nPeak gaming hours (16-23): {df['is_peak_gaming'].sum()} games")
print(f"Late night games (22-05): {df['is_night'].sum()} games")

Time distribution by period:
time_of_day
Afternoon    41505
Evening      33974
Morning      24306
Night        21547
Name: count, dtype: int64

Peak gaming hours (16-23): 56977 games
Late night games (22-05): 29967 games


In [68]:
import pandas as pd
import numpy as np

# 1. Elo rating imputation: median within the same time control
df["white_elo"] = df.groupby("time_control")["white_elo"].transform(
    lambda x: x.fillna(x.median())
)
df["black_elo"] = df.groupby("time_control")["black_elo"].transform(
    lambda x: x.fillna(x.median())
)

# Fallback to global median if any group median is still null (unlikely)
df["white_elo"] = df["white_elo"].fillna(df["white_elo"].median())
df["black_elo"] = df["black_elo"].fillna(df["black_elo"].median())

# 2. Rating diff imputation: 
#    - If both Elo and diff are missing, diff = 0 (no change) is reasonable
#    - If only diff missing but Elo present, use median diff from similar rating ranges
#    Simple version: fill with median diff per time control
df["white_rating_diff"] = df.groupby("time_control")["white_rating_diff"].transform(
    lambda x: x.fillna(x.median())
)
df["black_rating_diff"] = df.groupby("time_control")["black_rating_diff"].transform(
    lambda x: x.fillna(x.median())
)

# Final fallback to 0 (common for unrated/new players)
df["white_rating_diff"] = df["white_rating_diff"].fillna(0)
df["black_rating_diff"] = df["black_rating_diff"].fillna(0)

 13  eco                121332 non-null  str    
 14  opening            121332 non-null  str    
 15  time_control       121332 non-null  str    
 16  termination        121332 non-null  str    
 17  moves              121332 non-null  object 

In [71]:
df['eco'].value_counts()

eco
A00    9462
B00    6153
C00    5993
A40    5923
D00    5284
       ... 
A55       1
D75       1
E72       1
C04       1
B63       1
Name: count, Length: 411, dtype: int64

In [73]:
df['opening'].unique()

<ArrowStringArray>
[                                 'French Defense: Normal Variation',
                                   'Modern Defense: Geller's System',
                           'Queen's Gambit Declined: Normal Defense',
                            'Nimzo-Larsen Attack: English Variation',
                              'Nimzo-Larsen Attack: Spike Variation',
                    'Ruy Lopez: Berlin Defense, Rio Gambit Accepted',
                             'English Opening: Anglo-Indian Defense',
                                   'Scandinavian Defense: Main Line',
                                             'Three Knights Opening',
                                               'Nimzo-Larsen Attack',
 ...
             'French Defense: Rubinstein Variation, Kasparov Attack',
    'Sicilian Defense: Snyder Variation, Queen Fianchetto Variation',
               'Alekhine Defense: Modern Variation, Panov Variation',
                         'King's Indian Defense: Exchange Variatio

In [75]:
df['time_control'].unique()

['Classical', 'Bullet', 'Blitz', 'Correspondence']
Categories (4, str): ['Bullet' < 'Blitz' < 'Classical' < 'Correspondence']

In [77]:
df['termination'].unique()

<ArrowStringArray>
['Normal', 'Time forfeit']
Length: 2, dtype: str

In [78]:
df['moves']

0         [e2e4, e7e6, d2d4, b7b6, a2a3, c8b7, b1c3, g8h...
1         [e2e4, g7g6, g1f3, f8g7, c2c3, d7d6, d2d4, g8f...
2         [d2d4, d7d5, c2c4, g8f6, b1c3, e7e6, a2a3, f8e...
3         [b2b3, c7c5, c1b2, d7d6, d2d3, f7f6, b1d2, e7e...
4                [b2b3, g7g6, c1b2, g8f6, g2g4, b7b6, g4g5]
                                ...                        
121327    [d2d4, e7e6, e2e3, f8e7, g1f3, e7f6, g2g3, d7d...
121328    [e2e4, b7b6, d2d4, c8b7, d4d5, d7d6, b1c3, g7g...
121329    [e2e4, d7d5, e4d5, g8f6, b1c3, f6d5, c3d5, d8d...
121330    [e2e4, e7e5, g1f3, d7d6, f1c4, h7h6, d2d4, e5d...
121331    [e2e4, c7c6, d2d4, d7d5, b1c3, d5e4, c3e4, g8f...
Name: moves, Length: 121332, dtype: object